# 09. Final CNN vs ViT Comparison — Lab-to-Field Generalization

**Assignment core task:** fine-tune a CNN and a Vision Transformer on plant disease images; compare their accuracy on controlled laboratory images and real-world field images.

This notebook compares four **already-trained, fixed checkpoints** under a unified evaluation protocol. **No training or checkpoint selection takes place.**

**Main questions**
1. How do ResNet18 and ViT-Small perform on PlantVillage Lab Test?
2. How much do their predictions deteriorate on PlantDoc Field Dev?
3. Does mixed-domain adaptation improve the Lab–Field trade-off?

**Evaluation caveat:** Field Dev (`n=92`) was used to select the mixed-domain checkpoints. Field Dev metrics are *development results*, **not independent test evidence**. The original 47-image Field Test has been inspected during earlier experimentation, so it is not treated as a newly independent final test here. An additional external field dataset is required for confirmatory claims.

All notebook content and comments are in English for the research repository.

## 1. Setup

**Purpose:** connect to the current GitHub repository and the fixed checkpoints stored in Google Drive. Run on Colab GPU (T4) to make evaluation practical. This notebook intentionally does not update model weights.

In [ ]:
%pip install -q "timm==1.0.26" datasets pandas scikit-learn matplotlib pillow

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from google.colab import drive
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights, resnet18

REPO_URL = "https://github.com/thutrang147/plant-disease-diagnosis.git"
PROJECT_ROOT = Path("/content/plant-disease-diagnosis")
if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
GIT_SHA = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import PlantDiseaseDataset

drive.mount("/content/drive")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
DRIVE_ROOT = Path("/content/drive/MyDrive/plant-disease-diagnosis")
OUTPUT_ROOT = DRIVE_ROOT / "final_comparison_development"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository commit:", GIT_SHA)
print("Evaluation device:", DEVICE)
print("Output directory:", OUTPUT_ROOT)

## 2. Lock checkpoints and dataset splits

**Purpose:** make sure every model is evaluated using the same manifest and class mapping. The two adapted checkpoints were selected using Field Dev and must match its exact SHA-256 hash.

The final comparison uses:
- **Lab Test:** 1,692 PlantVillage images (the same original test manifest for all models).
- **Field Dev:** 92 PlantDoc images (near-duplicate-aware verified split).

The original Field Test is **not** used for model inference.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

LABEL_MAP = json.loads((SPLITS_DIR / "label_map.json").read_text(encoding="utf-8"))
CLASS_NAMES = [name for name, idx in sorted(LABEL_MAP.items(), key=lambda item: item[1])]
LABEL_IDS = list(range(5))
assert sorted(LABEL_MAP.values()) == LABEL_IDS and len(CLASS_NAMES) == 5

MANIFESTS = {
    name: SPLITS_DIR / f"{name}.csv"
    for name in ("lab_test", "lab_val", "field_dev", "field_adapt_train", "field_test")
}
for name, path in MANIFESTS.items():
    assert path.is_file(), f"Missing {name} manifest: {path}. Push the verified manifests to GitHub."
frames = {name: pd.read_csv(path) for name, path in MANIFESTS.items()}
assert len(frames["lab_test"]) == 1692
assert len(frames["field_dev"]) == 92
assert len(frames["field_adapt_train"]) == 370
assert len(frames["field_test"]) == 47
assert len(frames["lab_val"]) == 1341

for name, frame in frames.items():
    assert frame["row_id"].is_unique, f"Repeated source row_id: {name}"
    assert (frame["class_name"].map(LABEL_MAP) == frame["label_id"]).all(), name
    expected_source = "plantvillage" if name.startswith("lab_") else "plantdoc"
    assert (frame["source"] == expected_source).all(), name

# A manifest-level check; original image-content audit was completed in Steps 6–8.
assert set(frames["field_adapt_train"].row_id).isdisjoint(set(frames["field_dev"].row_id))
assert set(frames["field_adapt_train"].row_id).isdisjoint(set(frames["field_test"].row_id))
assert set(frames["field_dev"].row_id).isdisjoint(set(frames["field_test"].row_id))
assert set(frames["lab_val"].row_id).isdisjoint(set(frames["lab_test"].row_id))
for a, b in (("field_adapt_train", "field_dev"), ("field_adapt_train", "field_test"), ("field_dev", "field_test")):
    if "group_id" in frames[a].columns and "group_id" in frames[b].columns:
        assert set(frames[a].group_id).isdisjoint(set(frames[b].group_id)), (a, b)

MANIFEST_HASHES = {name: sha256_file(path) for name, path in MANIFESTS.items()}
print("Class order:", CLASS_NAMES)
print("Lab Test:", len(frames["lab_test"]), "Field Dev:", len(frames["field_dev"]))
print("Field Dev SHA256:", MANIFEST_HASHES["field_dev"])
print("Field Test is present but is NOT evaluated in this notebook.")
display(pd.crosstab(
    pd.concat([
        frames["lab_test"].assign(partition="Lab Test"),
        frames["field_dev"].assign(partition="Field Dev"),
    ], ignore_index=True)["class_name"],
    pd.concat([
        frames["lab_test"].assign(partition="Lab Test"),
        frames["field_dev"].assign(partition="Field Dev"),
    ], ignore_index=True)["partition"],
))

In [ ]:
CHECKPOINT_SPECS = {
    "ResNet18 · Lab baseline": {
        "architecture": "resnet18", "checkpoint": DRIVE_ROOT / "resnet18_baseline/resnet18_lab_best.pt",
        "source_checkpoint": DRIVE_ROOT / "resnet18_baseline/resnet18_lab_best.pt", "batch_size": 32,
        "phase": "lab_only",
    },
    "ViT-Small · Lab baseline": {
        "architecture": "vit_small_patch16_224.augreg_in1k",
        "checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt",
        "source_checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt", "batch_size": 16,
        "phase": "lab_only",
    },
    "ResNet18 · Geometric + Mixed": {
        "architecture": "resnet18",
        "checkpoint": DRIVE_ROOT / "mixed_domain_verified/R2_resnet_geometric/mixed_domain_best.pt",
        "source_checkpoint": DRIVE_ROOT / "resnet18_aug_geometric/resnet18_best.pt", "batch_size": 32,
        "phase": "mixed_domain",
    },
    "ViT-Small · Mixed": {
        "architecture": "vit_small_patch16_224.augreg_in1k",
        "checkpoint": DRIVE_ROOT / "mixed_domain_verified/V1_vit_small/mixed_domain_best.pt",
        "source_checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt", "batch_size": 16,
        "phase": "mixed_domain",
    },
}

for name, spec in CHECKPOINT_SPECS.items():
    assert spec["checkpoint"].is_file(), f"Missing checkpoint for {name}: {spec['checkpoint']}"
    assert spec["source_checkpoint"].is_file(), f"Missing source checkpoint for {name}: {spec['source_checkpoint']}"

print("All four fixed checkpoints are available.")
display(pd.DataFrame([
    {"model": name, "phase": spec["phase"], "checkpoint": str(spec["checkpoint"])}
    for name, spec in CHECKPOINT_SPECS.items()
]))

## 3. Reconstruct each architecture's exact evaluation preprocessing

**Purpose:** CNN and ViT were trained using different pretrained-image preprocessing configurations. Reusing each model's original evaluation transform prevents an accidental and unfair preprocessing mismatch.

For mixed-domain checkpoints, ViT preprocessing is recovered from the **source Lab checkpoint**, because the mixed checkpoint stores the weights but not the full timm preprocessing dictionary.

In [ ]:
def load_fixed_model(name: str):
    spec = CHECKPOINT_SPECS[name]
    state = torch.load(spec["checkpoint"], map_location="cpu", weights_only=True)
    source = torch.load(spec["source_checkpoint"], map_location="cpu", weights_only=True)
    architecture = spec["architecture"]

    assert list(state["class_names"]) == CLASS_NAMES, f"Checkpoint label mapping mismatch: {name}"
    assert state["architecture"] == architecture, f"Checkpoint architecture mismatch: {name}"
    assert list(source["class_names"]) == CLASS_NAMES, f"Source label mapping mismatch: {name}"
    assert source["architecture"] == architecture

    if spec["phase"] == "mixed_domain":
        assert state.get("adaptation_type") == "balanced_mixed_domain", name
        assert state["field_dev_sha256"] == MANIFEST_HASHES["field_dev"], (
            f"Different Field Dev split for checkpoint: {name}"
        )
        assert state["manifest_sha256"]["lab_val"] == MANIFEST_HASHES["lab_val"], name
        assert state["source_checkpoint_sha256"] == sha256_file(spec["source_checkpoint"]), (
            f"Source checkpoint mismatch: {name}"
        )

    if architecture == "resnet18":
        model = resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
        transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
    else:
        assert architecture == "vit_small_patch16_224.augreg_in1k"
        config = source["preprocessing_config"]
        assert tuple(config["input_size"]) == (3, 224, 224)
        model = timm.create_model(architecture, pretrained=False, num_classes=len(CLASS_NAMES))
        transform = timm.data.create_transform(**config, is_training=False)

    model.load_state_dict(state["model_state_dict"], strict=True)
    model.to(DEVICE).eval()
    return model, transform


def predict(model, loader):
    observed, predicted, confidence = [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(DEVICE, non_blocking=(DEVICE.type == "cuda")))
            prob, prediction = torch.softmax(logits.float(), dim=1).max(dim=1)
            observed.extend(labels.tolist())
            predicted.extend(prediction.cpu().tolist())
            confidence.extend(prob.cpu().tolist())
    return np.asarray(observed), np.asarray(predicted), np.asarray(confidence)


def metrics(observed, predicted):
    return {
        "accuracy": float(accuracy_score(observed, predicted)),
        "macro_f1": float(f1_score(observed, predicted, labels=LABEL_IDS,
                                   average="macro", zero_division=0)),
        "correct": int(np.count_nonzero(observed == predicted)),
        "images": int(len(observed)),
    }

## 4. Run inference on the two fixed evaluation sets

**Purpose:** calculate Accuracy and Macro-F1 from fresh predictions, rather than copying scores from previous notebooks. Each checkpoint is evaluated on **exactly the same 1,692 Lab Test and 92 Field Dev images**.

**No optimizer, training, tuning, or Field Test inference is performed.**

In [ ]:
rows = []
per_class_rows = []
prediction_frames = {}

for model_name, spec in CHECKPOINT_SPECS.items():
    print("\nEvaluating:", model_name)
    model, transform = load_fixed_model(model_name)
    model_key = list(CHECKPOINT_SPECS).index(model_name) + 1
    for partition in ("lab_test", "field_dev"):
        dataset = PlantDiseaseDataset(MANIFESTS[partition], transform=transform)
        loader = DataLoader(
            dataset, batch_size=spec["batch_size"], shuffle=False,
            num_workers=0, pin_memory=(DEVICE.type == "cuda"),
        )
        assert dataset.metadata.row_id.tolist() == frames[partition].row_id.tolist()
        observed, predicted, confidence = predict(model, loader)
        assert np.array_equal(observed, frames[partition].label_id.to_numpy())

        score = metrics(observed, predicted)
        rows.append({
            "model": model_name, "phase": spec["phase"], "partition": partition,
            **score,
        })

        report = classification_report(
            observed, predicted, labels=LABEL_IDS, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        )
        for disease in CLASS_NAMES:
            per_class_rows.append({
                "model": model_name, "partition": partition, "class_name": disease,
                "support": int(report[disease]["support"]),
                "precision": float(report[disease]["precision"]),
                "recall": float(report[disease]["recall"]),
                "f1": float(report[disease]["f1-score"]),
            })

        result = frames[partition][["source", "row_id", "class_name", "label_id"]].copy()
        result["predicted_label_id"] = predicted
        result["predicted_class"] = [CLASS_NAMES[int(i)] for i in predicted]
        result["confidence"] = confidence
        result["correct"] = observed == predicted
        result["model"] = model_name
        result["partition"] = partition
        prediction_frames[(model_name, partition)] = result
        result.to_csv(OUTPUT_ROOT / f"predictions_model{model_key}_{partition}.csv", index=False)

        cm = confusion_matrix(observed, predicted, labels=LABEL_IDS)
        fig, ax = plt.subplots(figsize=(9, 7))
        ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
            ax=ax, colorbar=False, cmap="Blues", xticks_rotation=30
        )
        ax.set_title(f"{model_name} — {partition} (n={len(observed)})")
        fig.tight_layout()
        fig.savefig(OUTPUT_ROOT / f"confusion_model{model_key}_{partition}.png", dpi=160)
        plt.show()
        plt.close(fig)
        print(f"  {partition}: Accuracy={score['accuracy']:.4f} | "
              f"Macro-F1={score['macro_f1']:.4f} | Correct={score['correct']}/{score['images']}")
        del dataset, loader
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

summary = pd.DataFrame(rows)
per_class = pd.DataFrame(per_class_rows)
summary.to_csv(OUTPUT_ROOT / "final_comparison_summary.csv", index=False)
per_class.to_csv(OUTPUT_ROOT / "final_comparison_per_class.csv", index=False)
display(summary.round(4))

## 5. Compare in-domain and cross-domain performance

**Purpose:** compare Lab Test with Field Dev for each model, then inspect the macro-F1 drop. **This drop is descriptive**, because the two datasets have different image sources and sampling processes.

The adapted models were selected using Field Dev, so gains seen on this same set are not independent confirmation of improvement.

In [ ]:
wide = summary.pivot(index="model", columns="partition", values=["accuracy", "macro_f1"])
wide.columns = [f"{metric}_{partition}" for metric, partition in wide.columns]
wide = wide.reindex(CHECKPOINT_SPECS.keys())
wide["macro_f1_lab_minus_field"] = wide["macro_f1_lab_test"] - wide["macro_f1_field_dev"]
wide.to_csv(OUTPUT_ROOT / "lab_field_tradeoff.csv")
display(wide.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
model_labels = ["ResNet18\nBaseline", "ViT-Small\nBaseline",
                "ResNet18\nGeometric+Mixed", "ViT-Small\nMixed"]
x = np.arange(len(model_labels))
bar_width = 0.37
for ax, metric, title in (
    (axes[0], "macro_f1", "Macro-F1"),
    (axes[1], "accuracy", "Accuracy"),
):
    lab = wide[f"{metric}_lab_test"].to_numpy()
    field = wide[f"{metric}_field_dev"].to_numpy()
    ax.bar(x - bar_width/2, lab, width=bar_width, label="Lab Test (n=1,692)")
    ax.bar(x + bar_width/2, field, width=bar_width, label="Field Dev (n=92)")
    ax.set_title(f"CNN vs ViT — {title}")
    ax.set_ylabel(title)
    ax.set_ylim(0, 1.06)
    ax.set_xticks(x, model_labels)
    ax.legend()
    ax.grid(axis="y", alpha=0.15)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "cnn_vs_vit_lab_field.png", dpi=160)
plt.show()
plt.close(fig)

## 6. Per-class Field Dev recall and paired errors

**Purpose:** identify which diseases remain difficult after adaptation. Paired errors on the same images reveal where models fail together. These are diagnostic development analyses, not evidence of statistical superiority.

In [ ]:
field_recall = per_class.loc[per_class.partition == "field_dev"].pivot(
    index="class_name", columns="model", values="recall"
).reindex(index=CLASS_NAMES, columns=CHECKPOINT_SPECS.keys())
display(field_recall.round(4))

fig, ax = plt.subplots(figsize=(13, 5.5))
field_recall.plot(kind="bar", ax=ax, rot=23)
ax.set_title("PlantDoc Field Dev — Per-class recall (n=92)")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1.06)
ax.set_xlabel("Disease")
ax.grid(axis="y", alpha=0.15)
ax.legend(title="Checkpoint", fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "field_dev_per_class_recall.png", dpi=160)
plt.show()
plt.close(fig)

# Pair the two *adapted* model predictions using immutable row_id keys.
a = prediction_frames[("ResNet18 · Geometric + Mixed", "field_dev")]
b = prediction_frames[("ViT-Small · Mixed", "field_dev")]
paired = a[["row_id", "label_id", "class_name", "correct", "predicted_class"]].rename(
    columns={"correct": "resnet_correct", "predicted_class": "resnet_prediction"}
).merge(
    b[["row_id", "label_id", "correct", "predicted_class"]].rename(
        columns={"correct": "vit_correct", "predicted_class": "vit_prediction"}
    ), on=["row_id", "label_id"], how="inner", validate="one_to_one"
)
assert len(paired) == len(frames["field_dev"])
paired["outcome"] = np.select(
    [paired.resnet_correct & paired.vit_correct,
     paired.resnet_correct & ~paired.vit_correct,
     ~paired.resnet_correct & paired.vit_correct],
    ["both_correct", "resnet_only", "vit_only"], default="both_wrong"
)
paired.to_csv(OUTPUT_ROOT / "adapted_models_paired_field_dev.csv", index=False)
display(paired["outcome"].value_counts().rename("images").to_frame())

## 7. Save a reproducibility record

**Purpose:** make the comparison inspectable later without relying on execution order or personal memory. The record includes checkpoint hashes, Git SHA, model-selection caveats, and split hashes.

In [ ]:
record = {
    "research_task": "Fine-grained tomato leaf disease diagnosis: CNN vs ViT, Lab-to-Field",
    "git_commit": GIT_SHA,
    "device": str(DEVICE),
    "framework_versions": {"torch": torch.__version__, "timm": timm.__version__},
    "class_names": CLASS_NAMES,
    "manifests_sha256": MANIFEST_HASHES,
    "checkpoints_sha256": {
        name: sha256_file(spec["checkpoint"])
        for name, spec in CHECKPOINT_SPECS.items()
    },
    "evaluation_partitions": {"lab_test": 1692, "field_dev": 92},
    "field_test_inference_performed": False,
    "model_selection_note": "Mixed models were selected on Field Dev; results are developmental, not independent test results.",
    "independent_field_test_note": "The original 47-image Field Test was previously inspected and is not used as a fresh independent holdout.",
    "next_validation": "Collect or designate new independent field images and audit near duplicates and labels before confirmatory evaluation.",
}
(OUTPUT_ROOT / "comparison_protocol.json").write_text(
    json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved to:", OUTPUT_ROOT)
print("Files:")
for p in sorted(OUTPUT_ROOT.iterdir()):
    if p.is_file():
        print(" -", p.name)

## 8. Interpretation and assignment conclusion

After running all cells, write the conclusions using **the newly computed metrics**, not the previous notebooks' numbers.

**Questions to answer in the report:**
1. Which architecture performs better on **Lab Test**, and by how much?
2. Which architecture performs better on **Field Dev**, and which classes cause the largest errors?
3. Does Lab-only training transfer well to PlantDoc?
4. What did geometric augmentation and mixed-domain adaptation change, and at what cost to Lab performance?
5. Which findings are only development observations, and what would require a new independent field test?

**Scientific limitations:** small and class-imbalanced Field Dev sample; model selection on Field Dev; incomplete independent field validation; PlantDoc label inconsistencies and near-duplicates that required quarantine; differing model capacities and fine-tuning protocols. The two architectures were not matched on parameter count or compute.

**Assignment scope:** This notebook completes the core CNN–ViT comparison and the lab-to-field analysis extension. Quantization and mobile deployment are optional extensions and are not performed here.